# Foundry Harness — Build Notebook

This is the **one notebook** the whole harness gets built in. Run cells top to bottom on a fresh Colab runtime; new roles (Indexer, Cartographer, Detector, ...) get appended as new sections at the bottom rather than as separate notebook files, so nothing later ever loses the environment an earlier section set up (installed packages, OpenAI key, in-progress SQLite database).

**Sections:**
1. **Setup** — clone, install, fetch the CodeGuard rule corpus, enter your OpenAI key. Run once per fresh runtime.
2. **Substrate** — finding store, work queue, budget governor. No LLM calls — proves the constitution's structural guarantees hold on their own, before any agent touches them.
3. *(Indexer, Cartographer, Detector, Triager, Coverage-Guide, Reporter, and the full assembled pipeline will each get their own section appended below as they're built.)*

## Setup

Clones the repo (if needed), installs dependencies, fetches the CodeGuard rule corpus, and captures your OpenAI key. Makes no LLM calls itself.

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/harshamore/FoundryHarnessDC.git"
REPO_DIR = "FoundryHarnessDC"
PROJECT_SUBDIR = "langchain"  # this project lives in a subdirectory of the repo

# A browser reload reconnects to the SAME running Colab kernel -- it does not
# restart it. If this cell already ran once this session, `project_path` is
# already set in the notebook's namespace and Path.cwd() now points *inside*
# the checkout (from the os.chdir below). Detecting "already cloned" from
# cwd alone would then take the "local dev checkout" branch below and skip
# the sync entirely -- which is exactly what silently produced a
# ModuleNotFoundError for a module that had since been added on GitHub, no
# matter how many times the cell was rerun. Handle "already ran this
# session" as its own explicit, always-syncing case, first.
if "project_path" in globals() and (project_path / "pyproject.toml").exists():
    repo_git_dir = project_path.parent  # .../FoundryHarnessDC (langchain is the subdir)
    if (repo_git_dir / ".git").exists():
        !git -C {repo_git_dir} pull --quiet --ff-only
else:
    cwd = Path.cwd()
    if (cwd / "pyproject.toml").exists():
        # Already inside the project directory (e.g. running locally from repo root).
        project_path = cwd
    elif (cwd.parent / "pyproject.toml").exists():
        # Running from notebooks/ inside an already-cloned checkout.
        project_path = cwd.parent
    elif (cwd / REPO_DIR).exists():
        # Repo was already cloned earlier -- pull the latest rather than
        # silently reusing a checkout that may predate sections added since.
        !git -C {REPO_DIR} pull --quiet --ff-only
        project_path = cwd / REPO_DIR / PROJECT_SUBDIR
    else:
        !git clone --quiet {REPO_URL}
        project_path = cwd / REPO_DIR / PROJECT_SUBDIR

os.chdir(project_path)

# Make `foundry` importable in *this* kernel right away, without depending on
# pip's editable-install mechanism (a .pth file that Python's `site` module
# normally only processes at interpreter startup). A common Jupyter/Colab
# gotcha is `pip install -e` "succeeding" while the package still isn't
# importable until a kernel restart -- this sidesteps that entirely.
src_path = str(project_path / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Working directory: {os.getcwd()}")
print(f"'src' on sys.path: {src_path in sys.path}")

In [ ]:
%pip install --quiet -e ".[dev]"

### Fetch the CodeGuard rule corpus

Clones `cosai-oasis/project-codeguard` at a pinned commit and vendors `sources/rules/{core,owasp}` into `data/codeguard/rules/`. This has nothing to do with the Claude Code plugin on your laptop — Colab can't see that, so the harness fetches its own copy. See `docs/CODEGUARD_INTEGRATION.md`.

In [ ]:
!python scripts/fetch_codeguard_rules.py

In [ ]:
from pathlib import Path

core = list(Path("data/codeguard/rules/core").glob("*.md"))
owasp = list(Path("data/codeguard/rules/owasp").glob("*.md"))
print(f"core: {len(core)} rules, owasp: {len(owasp)} rules")
assert len(core) > 0 and len(owasp) > 0

### OpenAI API key

Entered interactively via `getpass` — never written to a file, never committed. Later sections read this from the environment. This cell itself makes no OpenAI calls.

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
print("OPENAI_API_KEY is set:", bool(os.environ.get("OPENAI_API_KEY")))

### Setup sanity check

Confirms `foundry` is importable before moving on to the Substrate section below.

In [ ]:
import importlib.util
import os

spec = importlib.util.find_spec("foundry")
if spec is None:
    raise ModuleNotFoundError(
        "'foundry' is not importable. Two likely causes, in order of likelihood:\n"
        "  1) The %pip install cell above errored or hadn't finished "
        "-- scroll up and check its output for 'Successfully installed foundry-harness'.\n"
        "  2) This cell ran in a different kernel session than the setup cell above "
        "(e.g. after Runtime > Restart session) -- rerun the whole notebook top to bottom.\n\n"
        f"Current working directory: {os.getcwd()}\n"
        "(should be a '.../langchain' directory containing pyproject.toml and src/foundry/)"
    )

from foundry.substrate.budget import BudgetGovernor
from foundry.substrate.db import connect
from foundry.substrate.finding_store import Citation, FindingStore, fingerprint
from foundry.substrate.work_queue import WorkQueue

print(f"'foundry' package found at: {spec.origin}")
print("Setup complete. Continue to the Substrate section below.")

### Kernel freshness check

The clone cell above pulls the latest *files* when it detects an existing checkout — but Python caches imported modules by name (`sys.modules`) and never re-reads a module's source just because the underlying file changed. If `foundry.substrate.db` (or anything else) was already imported earlier in this kernel session, a pull updates the files on disk without updating the code already loaded in memory. That mismatch shows up later as a confusing error (e.g. `OperationalError: no such table`) instead of here, where the actual cause is clear. This cell catches it early by comparing what's on disk against what's actually loaded.

In [ ]:
import re
from pathlib import Path

from foundry.substrate import db as db_module

on_disk_tables = set(re.findall(r"CREATE TABLE IF NOT EXISTS (\w+)", Path(db_module.__file__).read_text()))
loaded_tables = set(re.findall(r"CREATE TABLE IF NOT EXISTS (\w+)", db_module.SCHEMA))

if on_disk_tables != loaded_tables:
    raise RuntimeError(
        "Stale code loaded in this kernel: the schema on disk currently has "
        f"tables {sorted(on_disk_tables)}, but what's actually loaded in "
        f"this kernel's memory only has {sorted(loaded_tables)}. This "
        "happens when foundry.* modules were imported earlier in this "
        "kernel session, before the latest code was pulled -- a git pull "
        "updates files, it does not update modules already cached in "
        "sys.modules.\n\n"
        "Fix: Runtime -> Disconnect and delete runtime (more reliable than "
        "'Restart session' -- guarantees a genuinely fresh process), "
        "reconnect, then Run all from the top, in order, without skipping "
        "cells."
    )
print(f"Loaded schema matches what's on disk: {sorted(loaded_tables)}")

## Observability (Galileo AI)

Optional, automatic-only tracing: one [Galileo](https://galileo.ai) callback, attached to every real `agent.invoke(...)` call from here on. DeepAgents already produces an LLM-call / tool-call / subagent-delegation hierarchy through LangChain's own callback system -- this section captures it, nothing more. No line in `src/foundry/substrate/`, `coverage/`, `reporter/`, `triager/`, or `detector/` changes for this; it's wired entirely at the invocation edges (`src/foundry/observability/galileo.py`).

Entirely opt-in and fails soft. Leave the next cell's prompt blank and every section below runs exactly as it did before this section existed, untraced -- `galileo_run_config()` returns `None` whenever tracing isn't configured, which LangChain treats identically to not passing `config=` at all. A misconfigured or unreachable Galileo account degrades to "no tracing" rather than breaking a detection run; unlike an invalid `OPENAI_API_KEY`, a Galileo problem is never allowed to raise here, since tracing is peripheral instrumentation, not the harness's actual job.

In [ ]:
import getpass
import os

if not os.environ.get("GALILEO_API_KEY"):
    entered = getpass.getpass("Galileo API key (optional -- press Enter to skip tracing): ")
    if entered:
        os.environ["GALILEO_API_KEY"] = entered
print("GALILEO_API_KEY is set:", bool(os.environ.get("GALILEO_API_KEY")))

# The Setup section's install cell only installs the base ".[dev]" extras --
# `galileo` lives in its own ".[observability]" extra specifically so it's
# never installed unless it's actually going to be used. If a key was
# entered, install it now, before build_galileo_callback() needs it below.
if os.environ.get("GALILEO_API_KEY"):
    try:
        import galileo  # noqa: F401
    except ImportError:
        print("Installing the Galileo SDK ('.[observability]')...")
        %pip install --quiet -e ".[observability]"

In [ ]:
import os

from foundry.observability.galileo import DEFAULT_PROJECT

# Same guard shape as the API key cell above: skip the prompt entirely if
# GALILEO_PROJECT is already set (e.g. pre-set via a Colab secret, or this
# cell already ran once this session) -- and note this is also the SDK's
# own env var, read by GalileoLogger itself if this notebook's
# `project=` argument weren't passed at all.
if not os.environ.get("GALILEO_PROJECT") and os.environ.get("GALILEO_API_KEY"):
    entered = input(
        f"Galileo project name (optional -- press Enter for '{DEFAULT_PROJECT}'): "
    ).strip()
    if entered:
        os.environ["GALILEO_PROJECT"] = entered
galileo_project = os.environ.get("GALILEO_PROJECT") or DEFAULT_PROJECT
print(f"Galileo project: {galileo_project}")

### Wiring it in

`build_galileo_callback()` gets-or-creates the `project` name entered above (or `DEFAULT_PROJECT` if left blank) and a fixed `log_stream` in Galileo (real network call, only attempted if a key was entered) and returns a configured callback, or `None`. Built once here, reused by every section below: each `agent.invoke(...)` call passes `config=galileo_run_config(galileo_callback, run_name="<role>")`, and each becomes its own trace in the same project/log stream, tagged with the role that produced it -- `indexer`, `cartographer`, `detector-rule-sweep`, `detector-exploratory`, `detector-directed`, `triager`, `coverage-guide`, `reporter`, `full-pipeline`.

In [ ]:
from foundry.observability.galileo import build_galileo_callback, console_url

galileo_callback = build_galileo_callback(project=galileo_project, log_stream="colab")

if galileo_callback is not None:
    print(f"Galileo tracing enabled. View traces at: {console_url(galileo_callback)}")
else:
    print("Galileo tracing disabled (no GALILEO_API_KEY, or Galileo was unreachable) -- continuing without it.")

## Substrate

The non-agent machinery every role will depend on: the finding store, the work queue, and the budget governor. No LLM calls in this section — the point is to see the constitution's structural guarantees (not prompt instructions) hold on their own, before any agent touches them.

The same assertions live in `tests/test_finding_store.py` if you'd rather run them as a suite (`pytest tests/ -v`) — this section is the same proofs, run interactively so you can see the state change at each step.

In [ ]:
import tempfile
import time
import threading
from pathlib import Path

from foundry.substrate.db import connect
from foundry.substrate.finding_store import Citation, FindingStore, fingerprint
from foundry.substrate.work_queue import WorkQueue
from foundry.substrate.budget import BudgetCaps, BudgetGovernor

db_path = Path(tempfile.mkdtemp()) / "foundry.sqlite3"
print(f"Using scratch database: {db_path}")

## Constitution VIII — Fingerprints Are Stable Under Edit

A finding's identity is `(normalized_path, symbol, vulnerability_class)` — never a line number or snippet. Re-queueing the same candidate after an unrelated edit (simulated here by changing only the description) must not create a duplicate.

In [ ]:
conn = connect(db_path)
store = FindingStore(conn)

id1, fp1, was_new1 = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="get_user_by_name",
    vulnerability_class="sql-injection",
    description="Detected on first sweep",
    technique="codeguard-rule:input-validation-injection",
)
print(f"First queue:  id={id1} fingerprint={fp1} was_new={was_new1}")

# Simulate a re-run after the function moved a few lines -- only the
# description text differs, identity fields are unchanged.
id2, fp2, was_new2 = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="get_user_by_name",
    vulnerability_class="sql-injection",
    description="Detected on second sweep, function now 4 lines lower",
    technique="codeguard-rule:input-validation-injection",
)
print(f"Second queue: id={id2} fingerprint={fp2} was_new={was_new2}")
assert id1 == id2 and not was_new2, "should have deduplicated, not re-filed"

## Constitution I — Evidence Over Assertion

`assign_verdict()` will not accept `true-positive` unless every citation resolves against a resolver. Here the resolver is a small fake symbol table standing in for the real Indexer (built in the Indexer section, appended later in this notebook) — the mechanism is identical either way. First: a clean pass. Then: a deliberately fabricated citation, to watch it get demoted rather than silently accepted.

In [ ]:
known_symbols = {"get_user_by_name", "users_endpoint", "read_uploaded_file", "files_endpoint"}

def fake_resolver(c: Citation) -> bool:
    return c.symbol in known_symbols

clean_citations = [
    Citation("data/toy_target/vulnerable_app.py", "users_endpoint", "reachability"),
    Citation("data/toy_target/vulnerable_app.py", "get_user_by_name", "impact"),
]
verdict = store.assign_verdict(id1, "true-positive", clean_citations, "clean investigation", fake_resolver)
print(f"Clean citations -> verdict: {verdict}")
assert verdict == "true-positive"

In [ ]:
id3, _, _ = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="read_uploaded_file",
    vulnerability_class="path-traversal",
    description="candidate",
    technique="exploratory",
)

fabricated_citations = [
    Citation("data/toy_target/vulnerable_app.py", "sanitize_path_properly", "reachability"),  # does not exist
    Citation("data/toy_target/vulnerable_app.py", "read_uploaded_file", "impact"),
]
verdict = store.assign_verdict(id3, "true-positive", fabricated_citations, "confident but wrong", fake_resolver)
print(f"Fabricated citation -> verdict: {verdict}")
assert verdict == "needs-review", "should have been demoted, not accepted as true-positive"

row = store.get(id3)
print("\nRecorded investigation report:\n", row["investigation_report"])

## Constitution IV — Claims Are Atomic And Mortal

Enqueue several tasks, then race multiple worker threads (each with its own SQLite connection, simulating separate agent processes) to claim them. No task should ever be claimed by more than one worker, and none should be lost.

In [ ]:
queue = WorkQueue(conn, lease_seconds=60)
n_tasks, n_workers = 25, 8
task_ids = {queue.enqueue("index_function", {"i": i}) for i in range(n_tasks)}

claimed_by: dict[int, list[str]] = {}
lock = threading.Lock()

def worker(worker_id: str) -> None:
    wconn = connect(db_path)
    wqueue = WorkQueue(wconn, lease_seconds=60)
    while True:
        task = wqueue.claim_next(worker_id, task_type="index_function")
        if task is None:
            break
        with lock:
            claimed_by.setdefault(task.id, []).append(worker_id)
        wqueue.release(task.id, worker_id, status="done")
    wconn.close()

threads = [threading.Thread(target=worker, args=(f"worker-{i}",)) for i in range(n_workers)]
[t.start() for t in threads]
[t.join(timeout=30) for t in threads]

assert set(claimed_by.keys()) == task_ids, "every task should be claimed exactly once, none lost"
assert all(len(w) == 1 for w in claimed_by.values()), "no task should ever be double-claimed"
print(f"{len(task_ids)} tasks, {n_workers} racing workers -> every task claimed exactly once. No duplicates, none lost.")

## Constitution III — Liveness By Heartbeat, Never By Clock

A claim is only reclaimable once its lease has expired -- never on a fixed wall-clock timeout. Below: a task claimed under a zero-second lease becomes immediately reclaimable; a task claimed under a normal lease does not.

In [ ]:
stale_queue = WorkQueue(conn, lease_seconds=0)  # lease expires immediately
stale_task_id = stale_queue.enqueue("probe", {})
claimed = stale_queue.claim_next("agent-a", task_type="probe")
time.sleep(1.1)

reclaimer_conn = connect(db_path)
reclaimer_queue = WorkQueue(reclaimer_conn, lease_seconds=60)
reclaimed = reclaimer_queue.claim_next("agent-b", task_type="probe")
print(f"Stale claim reclaimed by agent-b: {reclaimed is not None and reclaimed.id == stale_task_id}")
assert reclaimed is not None and reclaimed.id == stale_task_id

fresh_queue = WorkQueue(conn, lease_seconds=60)
fresh_task_id = fresh_queue.enqueue("probe", {})
fresh_queue.claim_next("agent-c", task_type="probe")
stolen = reclaimer_queue.claim_next("agent-d", task_type="probe")
print(f"Fresh claim stolen while agent-c still heartbeating: {stolen is not None and stolen.id == fresh_task_id}")
assert not (stolen is not None and stolen.id == fresh_task_id), "a live claim should not be reclaimable"

## Constitution VI — Coverage Before Yield

`should_stop()` is a conjunction: low yield alone never halts the fleet while coverage is incomplete. Three scenarios: incomplete coverage with zero yield (must not stop), complete coverage with low yield (must stop), complete coverage with healthy yield (must not stop).

In [ ]:
gov = BudgetGovernor(conn, BudgetCaps(yield_threshold=0.5))
gov.record_spend(100.0, "detector sweep so far")

stop, reason = gov.should_stop(coverage_complete=False)
print(f"Coverage incomplete, zero yield -> stop={stop} ({reason})")
assert stop is False, "must not stop on yield alone while coverage is incomplete"

stop, reason = gov.should_stop(coverage_complete=True)
print(f"Coverage complete, zero yield -> stop={stop} ({reason})")
assert stop is True

## Substrate section: recap

Every principle above held under a live workload, with concurrent threads standing in for a real multi-agent fleet — no LLM was involved anywhere in this section. The **Indexer** section comes next, appended below in this same notebook: the first real OpenAI-backed agent, reading `data/toy_target/vulnerable_app.py` and exposing it to the rest of the fleet through the query interface spec.md §5.2 requires.

## Indexer

The structural knowledge every other role queries: a function inventory and a call graph for the target, plus a query interface (spec.md FR-022) — `get_function_body`, `get_callers`, `get_callees`, `find_symbol`, `full_text_search`.

**FR-020 matters here**: the inventory must come from a deterministic parser, not solely an LLM. Python's own `ast` module plays that role (`src/foundry/indexer/parser.py`) — no model call happens anywhere in the indexing itself. The LLM only shows up at the end of this section, once there's a real index to query.

In [ ]:
from foundry.indexer.parser import index_file
from foundry.indexer.store import IndexStore

repo_root = project_path  # set by the Setup section above
target = repo_root / "data" / "toy_target" / "vulnerable_app.py"
normalized_path = str(target.resolve().relative_to(repo_root.resolve()))

result = index_file(target, repo_root)
print(f"Functions found ({len(result.functions)}):")
for fn in result.functions:
    print(f"  {fn.name}  (lines {fn.lineno}-{fn.end_lineno})")

print(f"\nDirect call edges found ({len(result.call_edges)}):")
for edge in sorted(set(result.call_edges), key=lambda e: (e.caller, e.callee)):
    print(f"  {edge.caller} -> {edge.callee}")

### Query interface (FR-022)

Persist the index, then exercise the same query interface every downstream role will use: `get_function_body`, `get_callers`, `get_callees`, `find_symbol`, `full_text_search`. Still no LLM — this is the deterministic interface the agent layer gets built on top of.

In [ ]:
index_store = IndexStore(conn)  # reuses the `conn` opened in the Substrate section above
index_store.write_index(normalized_path, result.functions, result.call_edges)

print("get_function_body('get_user_by_name'):")
print(index_store.get_function_body("get_user_by_name"))

print(f"\nget_callers('get_user_by_name'): {index_store.get_callers('get_user_by_name')}")
print(f"get_callees('users_endpoint'): {index_store.get_callees('users_endpoint')}")

sym = index_store.find_symbol("read_uploaded_file")
print(f"\nfind_symbol('read_uploaded_file'): {sym['file']} lines {sym['lineno']}-{sym['end_lineno']}")

print(f"\nfull_text_search('UPLOAD_DIR'): {index_store.full_text_search('UPLOAD_DIR')}")

# Re-indexing the same file is idempotent (FR-025/026) -- rerun this cell as
# many times as you like, the function count never grows.
print(f"\nTotal indexed functions: {len(index_store.list_functions(file=normalized_path))}")

### The real evidence-gate resolver

Back in the Substrate section, `assign_verdict()`'s evidence gate was demonstrated with `fake_resolver` — a small hand-typed set of "known" symbols standing in for the real Indexer. Now that a real index exists, swap it in: `index_store.symbol_exists(path, symbol)` checks against functions the parser actually found, not a hand-typed list. `assign_verdict()` itself is unchanged — this is exactly the dependency-inversion point the Substrate section set up for.

Same two cases as before, now grounded in real parsed code: a citation naming a function the parser actually found accepts; a citation naming a function that was never defined (`sanitize_path_properly`) still gets demoted.

In [ ]:
def real_resolver(c: Citation) -> bool:
    return index_store.symbol_exists(c.path, c.symbol)

id4, _, _ = store.queue_candidate(
    normalized_path=normalized_path,
    symbol="read_uploaded_file",
    vulnerability_class="path-traversal",
    description="candidate, now checked against the real index",
    technique="exploratory",
)

# Real citation: read_uploaded_file really is defined in the target.
verdict = store.assign_verdict(
    id4,
    "true-positive",
    [Citation(normalized_path, "files_endpoint", "reachability"),
     Citation(normalized_path, "read_uploaded_file", "impact")],
    "grounded in the real index",
    real_resolver,
)
print(f"Real citations against real index -> verdict: {verdict}")
assert verdict == "true-positive"

# Fabricated citation: this function was never defined anywhere in the target.
id5, _, _ = store.queue_candidate(
    normalized_path=normalized_path,
    symbol="get_user_by_name",
    vulnerability_class="sql-injection",
    description="candidate with a fabricated citation",
    technique="exploratory",
)
verdict = store.assign_verdict(
    id5,
    "true-positive",
    [Citation(normalized_path, "sanitize_path_properly", "reachability")],
    "still fabricated, now checked against real code instead of a fake list",
    real_resolver,
)
print(f"Fabricated citation against real index -> verdict: {verdict}")
assert verdict == "needs-review"

### The first real agent

Everything above ran with no model call. Now wrap the query interface as a DeepAgents `SubAgent` (`foundry.agents.indexer.build_indexer_subagent`) and give it to a small main agent via `create_deep_agent` — the same `task`-tool delegation pattern every later role (Cartographer, Detector, ...) will use.

This cell makes a real call to OpenAI using the key you entered in Setup. It costs a small, real amount (`gpt-5.6-luna`, OpenAI's cheapest current tool-calling model). If it errors with an authentication failure, re-run the Setup section's OpenAI key cell above.

In [ ]:
from deepagents import create_deep_agent

from foundry.agents._middleware import minimal_filesystem_middleware
from foundry.agents.indexer import build_indexer_subagent
from foundry.observability.galileo import galileo_run_config

indexer_subagent = build_indexer_subagent(index_store)

harness_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[indexer_subagent],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent. For any question about the target's "
        "code structure, delegate to the 'indexer' subagent rather than "
        "guessing -- it has tools grounded in a real parsed index. You have "
        "no useful filesystem access of your own; don't try to explore files "
        "directly, just delegate."
    ),
)

response = harness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the indexer, tell me: which function(s) call get_db, and "
            "does untrusted input reach get_user_by_name through any of the "
            "endpoint functions? Cite the specific functions involved."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="indexer"))
print(response["messages"][-1].content)

## Indexer section: recap

The function inventory and call graph came from a deterministic parser (FR-020) — no model call anywhere in the indexing itself. The evidence gate now runs against real parsed code instead of a hand-typed stand-in. And the query interface it exposes is usable by an actual OpenAI-backed DeepAgents subagent, delegated to through the same `task`-tool pattern every later role will use.

**Cartographer** is next: the security map (architecture, attack surface, trust boundaries) every role reasons against, appended below in this same notebook.

## Cartographer

The security map: architecture overview, attack-surface enumeration, trust-boundary map, data-flow description, and a threat model synthesizing the rest (spec.md FR-030–FR-034). Every downstream role reasons against this.

Unlike the Indexer, this section's real output is meant to be LLM-authored — there's no deterministic-parser requirement here. What's structural instead is **FR-036a**: *"an empty security map is a Cartographer failure, not graceful degradation."* Every section gets a mechanically-derived fallback value first, from `src/foundry/cartographer/fallback.py`, before any model runs — so the map is never empty regardless of what the agent produces.

In [ ]:
from foundry.cartographer.fallback import (
    fallback_architecture_overview,
    fallback_attack_surface,
    fallback_data_flows,
    fallback_threat_model,
    fallback_trust_boundaries,
)
from foundry.cartographer.store import SecurityMapStore

security_map = SecurityMapStore(conn)  # reuses the `conn` opened in the Substrate section

security_map.write_section(
    "architecture_overview", fallback_architecture_overview(normalized_path, index_store), source="fallback"
)
security_map.write_section(
    "attack_surface", fallback_attack_surface(normalized_path, index_store), source="fallback"
)
security_map.write_section("trust_boundaries", fallback_trust_boundaries(), source="fallback")
security_map.write_section("data_flows", fallback_data_flows(), source="fallback")
security_map.write_section("threat_model", fallback_threat_model(), source="fallback")

print(f"Security map complete (FR-036a): {security_map.is_complete()}")
print("\n" + security_map.digest())

### The real Cartographer

Now build the Cartographer as a DeepAgents `SubAgent` (`foundry.agents.cartographer.build_cartographer_subagent`) and let it actually read the target and author each section, overwriting the fallback content. It has the same read-only index tools as the Indexer subagent, plus one write tool per section.

This makes a real OpenAI call and costs a small real amount (`gpt-5.6-luna`). If it errors with an authentication failure, re-run the Setup section's OpenAI key cell.

In [ ]:
from foundry.agents._middleware import minimal_filesystem_middleware
from foundry.agents.cartographer import build_cartographer_subagent
from foundry.observability.galileo import galileo_run_config

cartographer_subagent = build_cartographer_subagent(security_map, index_store)

mapping_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[cartographer_subagent],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent. Delegate to the 'cartographer' "
        "subagent to produce the security map for the target. You have no "
        "useful filesystem access of your own; don't try to explore files "
        "directly, just delegate."
    ),
)

response = mapping_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the cartographer, read the target's functions and produce "
            "the full security map: architecture overview, attack-surface "
            "enumeration, trust-boundary map, data-flow description, and a "
            "threat model. Call all five write tools -- the target is small, "
            "so brief, accurate sections grounded in the actual code are "
            "fine. Confirm when done."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="cartographer"))
print(response["messages"][-1].content)

In [ ]:
from foundry.cartographer.store import SECTIONS

for section in SECTIONS:
    print(f"{section}: source={security_map.get_source(section)}")

print("\n" + security_map.digest())

## Cartographer section: recap

Every section had real content (the fallback) before any model call — FR-036a held structurally, not by hoping the agent behaves. If the agent above skipped a section or errored, that section's `source` still reads `fallback` rather than being empty. `security_map.digest()` (FR-035) is what later roles — starting with Detector's exploratory mode next — get front-loaded into their own prompt context.

**Detector** is next: rule-sweep (CodeGuard rules wired in as tools) and exploratory hunting, appended below in this same notebook.

## Detector

Produces candidate findings, breadth-first (spec.md FR-037/FR-040). Two complementary halves, "not alternatives... the two halves of a flywheel": **rule-sweep** checks every function against the CodeGuard corpus systematically; **exploratory** hunting reasons freely about this specific target's design to find what generic rules miss. Both write only through `queue_candidate` — Constitution II ("Surface Only What Survives"): detection is high-volume, low-precision by design, and nothing reaches a human until a Triager (not built yet) promotes it.

In [ ]:
from foundry.codeguard.loader import load_rules

rules = load_rules(project_path / "data" / "codeguard" / "rules", categories=("core",))
print(f"Loaded {len(rules)} core CodeGuard rules. Sample:")
for r in rules[:5]:
    print(f"  {r.rule_id}: {r.description}")

### Rule-sweep

Systematic: for every function, check it against the CodeGuard corpus. This makes a real OpenAI call and costs a small real amount (`gpt-5.6-luna`) — potentially several tool calls as the agent works through 5 functions and 23 rules, so this one may take longer and cost a bit more than the single-turn Indexer/Cartographer demos.

In [ ]:
from foundry.agents._middleware import minimal_filesystem_middleware
from foundry.agents.detector import build_detector_rule_sweep_subagent
from foundry.observability.galileo import galileo_run_config

rule_sweep_subagent = build_detector_rule_sweep_subagent(store, index_store, rules)

rule_sweep_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[rule_sweep_subagent],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent. Delegate to the 'detector-rule-sweep' "
        "subagent to check every function against the CodeGuard rule corpus. "
        "You have no useful filesystem access of your own; don't try to "
        "explore files directly, just delegate."
    ),
)

response = rule_sweep_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the detector-rule-sweep subagent, check every function in "
            "the index against the CodeGuard core rule corpus and queue a "
            "candidate for anything that plausibly violates a rule. Report "
            "how many candidates you queued and why."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="detector-rule-sweep"))
print(response["messages"][-1].content)

In [ ]:
candidates = conn.execute(
    "SELECT id, symbol, vulnerability_class, technique, description FROM findings WHERE verdict IS NULL ORDER BY id"
).fetchall()
print(f"{len(candidates)} candidate(s) queued so far (rule-sweep):")
for c in candidates:
    print(f"  #{c['id']} {c['symbol']} [{c['vulnerability_class']}] via {c['technique']}")
    print(f"      {c['description'][:150]}")

### Exploratory hunting

Free-form: no rule checklist, just the target's actual design plus the Cartographer's security-map digest, front-loaded directly into the prompt (FR-035) rather than fetched via a tool. Another real OpenAI call.

In [ ]:
from foundry.agents.detector import build_detector_exploratory_subagent
from foundry.observability.galileo import galileo_run_config

exploratory_subagent = build_detector_exploratory_subagent(
    store, index_store, security_map_digest=security_map.digest()
)

exploratory_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[exploratory_subagent],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent. Delegate to the 'detector-exploratory' "
        "subagent to freely hunt for vulnerabilities in the target. You have no "
        "useful filesystem access of your own; don't try to explore files "
        "directly, just delegate."
    ),
)

response = exploratory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the detector-exploratory subagent, hunt this target for "
            "vulnerabilities a generic rule checklist might miss -- reason "
            "about the specific design, not a checklist. Queue anything you "
            "find, and record a rule gap if you're confident something you "
            "found has no CodeGuard rule that would have caught it."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="detector-exploratory"))
print(response["messages"][-1].content)

In [ ]:
all_candidates = conn.execute(
    "SELECT id, symbol, vulnerability_class, technique, description FROM findings WHERE verdict IS NULL ORDER BY id"
).fetchall()
print(f"{len(all_candidates)} candidate(s) queued in total (rule-sweep + exploratory):")
for c in all_candidates:
    print(f"  #{c['id']} {c['symbol']} [{c['vulnerability_class']}] via {c['technique']}")

rule_gaps = store.list_rule_gaps()
print(f"\n{len(rule_gaps)} rule gap(s) recorded:")
for g in rule_gaps:
    print(f"  {g['vulnerability_class']}: {g['pattern']}")

## Detector section: recap

Every candidate above is internal — Constitution II held structurally the whole time: neither subagent has any tool that reaches a human or an issue tracker, only `queue_candidate`. High volume, low precision is fine and expected here; that's what the next role exists to fix.

**Triager** is next: investigates each candidate and assigns a verdict, gated on structural evidence (spec.md FR-052) — the evidence-gate mechanism the Substrate and Indexer sections already built and tested standalone finally gets wired into a live pipeline, appended below in this same notebook.

## Triager

The noise filter (spec.md §5.5): investigates each candidate the Detector queued and assigns a verdict, gated on structural evidence (FR-050–054). This section adds no new enforcement mechanism of its own — `FindingStore.assign_verdict()`'s evidence gate has been tested since the Substrate section, first against a hand-typed fake resolver, then the real Indexer-backed one. What's new here is a live agent finally calling it under real load, instead of a notebook cell asserting on it directly.

In [ ]:
untriaged = store.list_untriaged()
print(f"{len(untriaged)} candidate(s) awaiting triage:")
for r in untriaged:
    print(f"  #{r['id']} {r['symbol']} [{r['vulnerability_class']}] via {r['technique']}")

if not untriaged:
    print("\n(None yet -- run the Detector section's real agent cells above first if you haven't.)")

### The real Triager

Investigates every untriaged candidate and calls `assign_verdict` with citations. A citation naming a symbol that doesn't actually exist gets auto-demoted from `true-positive` to `needs-review`, regardless of how confident the model's reasoning sounds — that's the evidence gate holding under a real agent's real (possibly wrong) claims, not a hand-crafted test case. Real OpenAI call.

In [ ]:
from foundry.agents.triager import build_triager_subagent
from foundry.observability.galileo import galileo_run_config

triager_subagent = build_triager_subagent(store, index_store, security_map_digest=security_map.digest())

triager_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[triager_subagent],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent. Delegate to the 'triager' subagent "
        "to investigate and assign a verdict to every untriaged candidate. "
        "You have no useful filesystem access of your own; don't try to "
        "explore files directly, just delegate."
    ),
)

response = triager_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the triager, investigate every untriaged candidate and "
            "assign each one a verdict with citations and a real "
            "investigation report. Report how many you triaged and to what "
            "verdicts."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="triager"))
print(response["messages"][-1].content)

In [ ]:
triaged = conn.execute(
    "SELECT id, symbol, vulnerability_class, verdict, investigation_report "
    "FROM findings WHERE verdict IS NOT NULL ORDER BY id"
).fetchall()
print(f"{len(triaged)} finding(s) triaged:")
for r in triaged:
    print(f"\n#{r['id']} {r['symbol']} [{r['vulnerability_class']}] -> {r['verdict']}")
    print(f"    {r['investigation_report'][:300]}")

remaining = store.list_untriaged()
print(f"\n{len(remaining)} still untriaged.")

### FR-054, live: a verdict needs a reason

"A verdict without an investigation report MUST be rejected by the finding store." No LLM needed to prove this — the tool itself refuses a bare label, whether the caller is a careful agent or not.

In [ ]:
id6, _, _ = store.queue_candidate(
    normalized_path=normalized_path,
    symbol="get_db",
    vulnerability_class="code-quality",
    description="bare-verdict demo, not a real finding",
    technique="exploratory",
)

try:
    store.assign_verdict(id6, "true-positive", [], "", resolver=lambda c: True)
    print("UNEXPECTED: bare verdict was accepted")
except ValueError as e:
    print(f"Correctly rejected: {e}")

# The finding is untouched -- rejection doesn't silently record anything.
row = store.get(id6)
print(f"\nFinding #{id6} verdict is still: {row['verdict']!r}")

## Triager section: recap

The evidence gate held under a live agent's real citations, not just hand-crafted test cases — `true-positive` only survives when every cited symbol actually resolves against the real index, and a bare verdict with no reasoning is rejected outright. `false-positive`/`needs-review`/`not-applicable`/`code-quality` verdicts stay internal; only `true-positive` findings will ever reach the Reporter's output later, per Constitution II.

**Coverage-Guide** is next: translates the operator's goals into a checklist, tracks progress, and — together with the budget governor already built in the Substrate section — provides the "done" signal (coverage ∧ yield, Constitution VI), appended below in this same notebook.

## Coverage-Guide

Half of the "done" signal (spec.md §5.7): translates the operator's goals into a checklist, tracks credible attempts against it, declares coverage complete. Like `BudgetGovernor`, every MUST-level requirement here (FR-067 derive the checklist, FR-069 check off from evidence, FR-070 directed tasks, FR-071 the coverage-complete flag, FR-074 don't rebuild from scratch) is mechanical — "coverage measures attempt, not outcome" is an evidence check, not a judgment call. The one place an LLM helps is FR-073 (a SHOULD): a short remaining-work estimate for the operator.

In [ ]:
from foundry.coverage.store import CoverageStore

coverage_store = CoverageStore(conn)  # reuses the `conn` opened in the Substrate section

# The operator's stated evaluation goals -- FR-068 requires these to be
# real, not synthesized. Matches the vulnerability classes this toy target
# actually has, so the checklist has genuine gaps to close.
operator_goals = ["sql-injection", "path-traversal", "hardcoded-credentials"]
areas = index_store.list_functions(file=normalized_path)

open_count = coverage_store.build_checklist(
    areas=areas,
    goals=operator_goals,
    bar_template="A rule-sweep or exploratory Detector pass has checked {area} for {goal}, "
                  "and the result (finding or clean) is recorded.",
)
print(f"Checklist built: {len(areas)} areas x {len(operator_goals)} goals = {open_count} open items")

### Review cycle (FR-069)

Checks every open item against the finding store — no LLM needed. If you've run the Detector/Triager sections live earlier in this session, some items close right now from that real evidence.

In [ ]:
result = coverage_store.review_cycle()
print(f"Closed this cycle: {len(result['closed_this_cycle'])}")
print(f"Still open: {len(result['still_open'])}")
print(f"Coverage complete: {coverage_store.is_complete()}")

if coverage_store.closed_items():
    print("\nClosed items:")
    for r in coverage_store.closed_items():
        print(f"  {r['area']} / {r['goal']} (closed {r['closed_at']})")

### Directed tasks (FR-070)

Queues a task for every still-open item — the `WorkQueue` built and concurrency-tested in the Substrate section finally gets a real use beyond proving its own atomicity. A future Detector could `claim_next(task_type="directed_detection:...")` and act on exactly the gap this identifies; wiring that consumption loop up live is out of scope for this toy, single-agent-invocation notebook, but the queue side is real.

In [ ]:
queued = coverage_store.queue_directed_tasks(queue)  # reuses the `queue` WorkQueue from the Substrate section
print(f"Queued {queued} directed task(s) for still-open items.")

# Idempotent: running it again queues nothing new for the same gaps.
requeued = coverage_store.queue_directed_tasks(queue)
print(f"Re-running immediately queues: {requeued} (deduplicated, as expected)")

### FR-068, live: no goals, no checklist

"The Coverage-Guide MUST NOT invent goals." An LLM handed an empty goals file will happily synthesize plausible-sounding ones and declare them covered — this refuses at the code level, before any model is involved.

In [ ]:
empty_goals_coverage = CoverageStore(conn)
try:
    empty_goals_coverage.build_checklist(areas=[], goals=[], bar_template="{area}::{goal}")
    print("UNEXPECTED: empty goals were accepted")
except ValueError as e:
    print(f"Correctly rejected: {e}")

### Constitution VI, closed: the real budget governor, wired to a real coverage flag

Back in the Substrate section, `gov.should_stop()` was tested with `coverage_complete` typed in by hand — `True` or `False` on a line above the call. That was always a placeholder for exactly this: a real `coverage_complete` value derived from a real checklist against real evidence, not a hand-typed boolean. Same `gov` object, same method, real inputs on both sides now.

In [ ]:
real_coverage_complete = coverage_store.is_complete()
stop, reason = gov.should_stop(coverage_complete=real_coverage_complete)
print(f"coverage_store.is_complete(): {real_coverage_complete}")
print(f"gov.should_stop(coverage_complete={real_coverage_complete}) -> stop={stop} ({reason})")

### The real Coverage-Guide (FR-073)

Everything above needed no model at all. This last piece is a SHOULD, not a MUST: a short remaining-work estimate for the operator, grounded in the actual checklist state via `get_coverage_report` — the one tool this subagent has. Real OpenAI call.

In [ ]:
from foundry.agents.coverage_guide import build_coverage_guide_subagent
from foundry.observability.galileo import galileo_run_config

coverage_guide_subagent = build_coverage_guide_subagent(coverage_store)

coverage_guide_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[coverage_guide_subagent],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent. Delegate to the 'coverage-guide' "
        "subagent for a coverage status estimate. You have no useful "
        "filesystem access of your own; don't try to explore files "
        "directly, just delegate."
    ),
)

response = coverage_guide_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the coverage-guide, get the current coverage report and "
            "give me a short estimate of remaining work with a one-line "
            "basis."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="coverage-guide"))
print(response["messages"][-1].content)

## Coverage-Guide section: recap

Coverage-Guide adds no new enforcement primitive — it's the same mechanical-conjunction pattern `BudgetGovernor` has used since the Substrate section, applied to a checklist instead of a spend/yield pair. Both halves of Constitution VI ("coverage AND yield, never yield alone") are now real: `coverage_store.is_complete()` feeds `gov.should_stop()` directly, no hand-typed booleans left anywhere in the loop.

**Reporter** is next, and last of the eight core roles: produces the human-facing output — per-finding writeups and an evaluation rollup — for `true-positive` findings only, appended below in this same notebook.

## Reporter

The last of the eight core roles (spec.md §5.8): produces the human-facing output — a self-contained writeup per confirmed finding, plus an evaluation rollup. Constitution II reaches its endpoint here: `publish_finding_report` refuses anything whose verdict isn't `true-positive` (FR-079), checked against the finding store itself, not the model's say-so. FR-083 — no naming the model, provider, or internal identifiers in a report meant for an external reviewer — is enforced the same way: a denylist scan on the actual text, not a prompt instruction hoping the model remembers.

This build's scope: local markdown files, not a real issue tracker (see `docs/ARCHITECTURE.md`). One file per finding, keyed by fingerprint (Constitution VIII) — republishing overwrites, never duplicates, the same idempotency FR-078/080 ask for.

In [ ]:
from foundry.reporter.store import ReporterStore

reporter_store = ReporterStore(conn, project_path / "data" / "reports")

eligible = store.list_by_verdict("true-positive")
print(f"{len(eligible)} true-positive finding(s) eligible for publication:")
for r in eligible:
    print(f"  #{r['id']} {r['symbol']} [{r['vulnerability_class']}] exploited={bool(r['exploited'])}")

if not eligible:
    print("\n(None yet -- run the Triager section's real agent cell above first if you haven't.)")

### The real Reporter

Publishes a self-contained report for every true-positive finding. If the model tries to mention itself, the model's provider, or this system's internals, `publish_finding_report` rejects the whole report — the notebook cell after this one proves that live. Real OpenAI call.

In [ ]:
from foundry.agents.reporter import build_reporter_subagent
from foundry.observability.galileo import galileo_run_config

reporter_subagent = build_reporter_subagent(store, reporter_store, index_store)

reporter_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[reporter_subagent],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent. Delegate to the 'reporter' "
        "subagent to publish a report for every true-positive finding. "
        "You have no useful filesystem access of your own; don't try to "
        "explore files directly, just delegate."
    ),
)

response = reporter_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the reporter, publish a self-contained report for every "
            "true-positive finding, with an appropriate severity and "
            "weakness classification for each. Report how many you "
            "published."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="reporter"))
print(response["messages"][-1].content)

In [ ]:
published = reporter_store.list_published()
print(f"{len(published)} report(s) published to {reporter_store.output_dir}:")
for r in published:
    print(f"  {r['finding_fingerprint']}: severity={r['severity']} weakness={r['weakness_class']}")
    print(f"    -> {r['report_path']}")

### FR-079 and FR-083, live: two ways to get rejected

No LLM needed for either — both are checked in code before a single byte gets written to disk.

In [ ]:
# FR-079: a needs-review finding can never be published, no matter how it's phrased.
id7, _, _ = store.queue_candidate(
    normalized_path=normalized_path,
    symbol="get_db",
    vulnerability_class="code-quality",
    description="FR-079 rejection demo, not a real finding",
    technique="exploratory",
)
store.assign_verdict(id7, "needs-review", [], "not enough evidence either way", resolver=lambda c: True)

try:
    reporter_store.publish_finding_report(id7, "Suspicious pattern", "Body.", "low", None)
    print("UNEXPECTED: needs-review was published")
except ValueError as e:
    print(f"FR-079 correctly rejected: {e}")

# FR-083: even a genuinely true-positive finding (earned honestly, via the
# real resolver) gets rejected if the report text leaks internals.
id8, _, _ = store.queue_candidate(
    normalized_path=normalized_path,
    symbol="get_db",
    vulnerability_class="code-quality",
    description="FR-083 rejection demo, not a real finding",
    technique="exploratory",
)
store.assign_verdict(
    id8,
    "true-positive",
    [Citation(normalized_path, "get_db", "impact")],
    "a real true-positive for this demo, verdict earned via the real resolver",
    resolver=real_resolver,
)

try:
    reporter_store.publish_finding_report(
        id8, "Title", "Detected using OpenAI's gpt-5.6-luna model.", "low", None
    )
    print("UNEXPECTED: report naming the model was published")
except ValueError as e:
    print(f"FR-083 correctly rejected: {e}")

### The rollup (FR-081)

Counts by severity and exploited status, findings grouped by component, and coverage status against the goals set in the Coverage-Guide section — entirely deterministic aggregation, no LLM needed.

In [ ]:
rollup = reporter_store.build_rollup(coverage_store)
print(rollup)

## Reporter section: recap — the eight core roles, complete

That's all eight roles from spec.md §4.2: Orchestrator (this notebook's own `create_deep_agent` calls, one per section, stand in for it — no dedicated Orchestrator subagent was built), Indexer, Cartographer, Detector, Triager, Validator (degraded — no testbed, `exploited` never set), Coverage-Guide, and Reporter. Every structural guarantee this build set out to prove — the evidence gate, atomic claims, heartbeat liveness, stable fingerprints, the coverage∧yield conjunction, Constitution II end to end from candidate to published report — has been demonstrated live against a real target with a real model, not just asserted in pytest.

**What's next**: a Full Pipeline section — wiring every subagent into one `create_deep_agent(...)` call instead of one per role — and a real closed loop for Coverage-Guide's directed tasks (a live Detector actually consuming `WorkQueue` gaps instead of the queue sitting unread), which was deliberately deferred earlier in this build specifically to come back to.

## Full Pipeline

Two things left, both deliberately deferred from earlier sections to be closed here together.

**Closing Coverage-Guide's directed-task loop (FR-070).** The Coverage-Guide section queued a directed task for every still-open checklist item, but nothing ever consumed them — the queue sat unread, and the section said so explicitly. Below, a real `detector-directed` subagent claims those tasks one at a time, investigates each named (area, goal) gap, and completes it — closing the checklist item for real, not just draining the queue (more on why that distinction needed its own fix, below).

**One agent, every role.** Every section so far built its own single-subagent `create_deep_agent(...)` call — clean for demonstrating one role at a time, but not what an actual Orchestrator looks like. Below, all eight subagents built across this notebook get wired into one `create_deep_agent(...)` call, and the main agent picks which one to delegate to per request via the same `task` tool every earlier section already used, just now with a real choice to make.

### Closing the directed-task loop, for real (FR-070)

A subtlety surfaced building this: just draining the work queue isn't the same as closing the coverage checklist. `CoverageStore.review_cycle()` closes an item on evidence — a `findings` row or a `coverage_log` sweep matching that exact (area, goal) — not on work-queue status. A directed pass that checks an area and finds nothing would previously drain its task with no effect on coverage at all, leaving the loop only superficially closed.

Fixed at the tool layer, not the prompt layer: `complete_directed_task` (`src/foundry/detector/tools.py`) now always calls `CoverageStore.record_sweep()` when a task is completed, using the *claimed task's own* area/goal — tracked server-side, not re-supplied by the model — regardless of whether `queue_candidate` was also called. "Coverage measures attempt, not outcome" (FR-069) is enforced the same way the Triager's evidence gate is: the tool decides what counts as evidence, the model just supplies what it found.

First, make sure there's something real to consume — re-running `queue_directed_tasks` is idempotent (proven in the Coverage-Guide section), so this is safe to run regardless of what happened earlier in this session.

In [ ]:
requeued = coverage_store.queue_directed_tasks(queue)  # idempotent -- only queues for items still open
pending_directed = conn.execute(
    "SELECT COUNT(*) AS n FROM work_queue WHERE task_type LIKE 'directed_detection:%' AND status = 'pending'"
).fetchone()["n"]

print(f"Open checklist items: {len(coverage_store.open_items())}")
print(f"Directed tasks queued just now: {requeued}")
print(f"Directed tasks pending in the work queue: {pending_directed}")

if pending_directed == 0:
    print("\n(Nothing pending -- coverage is already complete from earlier sections' real detection. "
          "That's the loop having nothing left to do, not a skipped step.)")

### The real directed Detector

`build_detector_directed_subagent` (`src/foundry/agents/detector.py`) gets the same read-only index tools as the other two Detector halves, plus `claim_directed_task`/`complete_directed_task`. Its instructions: claim one task, investigate it exactly like rule-sweep/exploratory would, queue a candidate if something's there, complete the task with a note either way, then claim the next one — until none remain.

Real OpenAI call, and potentially several tool calls in a loop if more than one directed task is pending — this may take longer than the single-shot Indexer/Cartographer demos, closer to the rule-sweep section's shape.

In [ ]:
from foundry.agents.detector import build_detector_directed_subagent
from foundry.observability.galileo import galileo_run_config

detector_directed_subagent = build_detector_directed_subagent(store, index_store, queue, coverage_store)

directed_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[detector_directed_subagent],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent. Delegate to the 'detector-directed' "
        "subagent to work through every directed-detection task Coverage-Guide "
        "has queued, one at a time, until none remain. You have no useful "
        "filesystem access of your own; don't try to explore files directly, "
        "just delegate."
    ),
)

response = directed_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Using the detector-directed subagent, claim and investigate "
            "every directed-detection task in the queue until none remain. "
            "For each one: investigate the named area and goal, queue a "
            "candidate if you find something, then always complete the task "
            "with a short note on what you actually checked. Report how "
            "many tasks you processed and how many candidates you found."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="detector-directed"))
print(response["messages"][-1].content)

In [ ]:
remaining_directed = conn.execute(
    "SELECT COUNT(*) AS n FROM work_queue WHERE task_type LIKE 'directed_detection:%' AND status = 'pending'"
).fetchone()["n"]
print(f"Directed tasks still pending: {remaining_directed}")

result = coverage_store.review_cycle()
print(f"Closed this cycle: {len(result['closed_this_cycle'])}")
print(f"Still open: {len(result['still_open'])}")
print(f"Coverage complete: {coverage_store.is_complete()}")

stop, reason = gov.should_stop(coverage_complete=coverage_store.is_complete())
print(f"gov.should_stop(coverage_complete={coverage_store.is_complete()}) -> stop={stop} ({reason})")

### One agent, every role

Every subagent built in this notebook — `indexer`, `cartographer`, `detector-rule-sweep`, `detector-exploratory`, `detector-directed`, `triager`, `coverage-guide`, `reporter` — registered on a single `create_deep_agent(...)` call. Nothing about any individual subagent changes: Constitution II still holds exactly as it did per-role, since it's each subagent's own `tools` list (not anything about how many subagents share one main agent) that keeps detection/triage internal and only the Reporter's tools human-facing.

The main agent now has a real choice to make on every request — which of eight descriptions best matches what's being asked — rather than one subagent being the only option. That costs more context per turn than any single-role call above, so the demo below is deliberately bounded: it asks for a coverage status and a report summary (two different subagents, no re-running the expensive multi-tool-call detection/triage work already done earlier in this session), rather than re-driving the whole pipeline through one prompt.

In [ ]:
from langgraph.graph.state import CompiledStateGraph

from foundry.observability.galileo import galileo_run_config

full_pipeline_agent = create_deep_agent(
    model="openai:gpt-5.6-luna",
    subagents=[
        indexer_subagent,
        cartographer_subagent,
        rule_sweep_subagent,
        exploratory_subagent,
        detector_directed_subagent,
        triager_subagent,
        coverage_guide_subagent,
        reporter_subagent,
    ],
    middleware=[minimal_filesystem_middleware()],
    system_prompt=(
        "You are the harness main agent, running the full fleet: indexer, "
        "cartographer, detector-rule-sweep, detector-exploratory, "
        "detector-directed, triager, coverage-guide, and reporter. Delegate "
        "every request to the specific subagent best suited to it -- never "
        "answer directly or explore files yourself, you have no useful "
        "filesystem access of your own."
    ),
)
print(f"All eight subagents wired into one agent: {isinstance(full_pipeline_agent, CompiledStateGraph)}")

response = full_pipeline_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Do two things, in order, delegating each to the specific "
            "subagent best suited to it: (1) using coverage-guide, get the "
            "current coverage report and state whether coverage is "
            "complete; (2) using reporter, list the currently published "
            "reports and give me a one-paragraph executive summary of the "
            "evaluation grounded in those actual counts. Don't re-run "
            "detection or triage -- that work is already done."
        ),
    }]
}, config=galileo_run_config(galileo_callback, run_name="full-pipeline"))
print(response["messages"][-1].content)

## Full Pipeline section: recap — this build's scope, complete

Both deferred pieces are closed. The directed-detection loop is real end to end: `queue_directed_tasks` writes claimable work, `detector-directed` claims and investigates it, and `complete_directed_task` leaves permanent evidence (`coverage_log`, via the tool layer, not the model's word for it) that flips the matching checklist item from open to closed on the next review cycle — proven both live above and in `tests/test_detector.py::test_complete_directed_task_closes_the_matching_coverage_checklist_item`. And all eight core-role subagents now run under one `create_deep_agent(...)` call, the actual shape an Orchestrator wires up, rather than one call standing in per section.

**What this doesn't claim.** Wiring eight subagents into one agent doesn't by itself exercise Constitution V (the Provider Is The Rate Arbiter) any more deeply than earlier sections did — that needs a genuinely concurrent workload (multiple subagent instances actually running at once), not just multiple subagents being *available* in one turn; if DeepAgents happens to dispatch two subagents' tool calls in parallel within a single turn, that's real concurrency sharing the same SQLite connection, protected by the same `lock_for` mechanism proven under test since the Substrate section — but nothing here forces or verifies that it happens. Constitution X (the Operator Outranks Every Agent) also remains not built: no `interrupt_on` gate exists on any tool, and no dedicated Orchestrator subagent was written — this notebook's own sequence of `create_deep_agent` calls has stood in for that role throughout, deliberately, per this build's scope.

That's the harness as scoped: eight core roles, source-only, Validator degraded (no testbed, FR-066), every MUST-level structural guarantee enforced in code and demonstrated live against a real target with a real model — not just asserted in tests.